In [5]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 50.4 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
import requests
import os

# GitHub API URL for the folder
url = "https://api.github.com/repos/ErikaJacobs/Harry-Potter-Text-Mining/contents/Book%20Text"

# Create directory
os.makedirs("harry_potter_books", exist_ok=True)

# Get file list
response = requests.get(url)
files = response.json()

# Download only .txt files
for file in files:
    if file['name'].endswith('.txt'):
        download_url = file['download_url']
        content = requests.get(download_url).text
        
        with open(f"harry_potter_books/{file['name']}", "w", encoding="utf-8") as f:
            f.write(content)

print("All books downloaded!")

All books downloaded!


In [2]:
folder_path = "harry_potter_books"

In [3]:
file_list = sorted([f for f in os.listdir(folder_path) if f.endswith(".txt")])

In [4]:
import os
import pandas as pd

# Folder where files are downloaded

all_dfs = []

# Loop through all txt files
for file_name in file_list:
    if file_name.endswith(".txt"):
        file_path = os.path.join(folder_path, file_name)
        # Read the file assuming separator is '@' and columns are already consistent
        df = pd.read_csv(file_path, sep='@')
        all_dfs.append(df)

# Concatenate all DataFrames
df = pd.concat(all_dfs, ignore_index=True)

In [31]:
df.columns = df.columns.str.lower()

In [ ]:
import re

def clean_text(text):
    text = str.lower(text)
    
    text = text.replace("\\'", "'").replace("\n", " ").replace("\r", " ")
        # Remove unwanted symbols (keep letters, numbers, and basic punctuation)
    text = re.sub(r"[^a-zA-Z0-9.,!?'\s]", '', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df.text.apply(clean_text)


In [34]:
import pandas as pd
import numpy as np
import spacy
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# Load spacy English tokenizer
nlp = spacy.load('en_core_web_md')

#### Using sentence-level tokenization is generally better for semantic quality, because you don’t want words from one scene or paragraph to influence unrelated words in another scene.

In [ ]:

# Step 1: Sentence-level tokenization and word cleaning
def tokenize_sentences(text):
    doc = nlp(text)  
    sentences = []
    for sent in doc.sents:
        tokens = [token.text for token in sent if token.is_alpha]
        if len(tokens) > 0:  
            sentences.append(tokens)
    return sentences

all_sentences = []
for text in df['text']:
    all_sentences.extend(tokenize_sentences(text))

print(f"Total number of sentences: {len(all_sentences)}")
print(f"Example sentence tokens: {all_sentences[0]}")

Total number of sentences: 74933
Example sentence tokens: ['the', 'boy', 'who', 'lived', 'mr', 'and', 'mrs', 'dursley', 'of', 'number', 'four', 'privet', 'drive', 'were', 'proud', 'to', 'say', 'that', 'they', 'were', 'perfectly', 'normal', 'thank', 'you', 'very', 'much']


In [ ]:
from collections import Counter

# Flatten all sentences to get word frequencies
all_words = [word for sentence in all_sentences for word in sentence]

# Count word frequencies
word_counts = Counter(all_words)
print(f"Total unique words: {len(word_counts)}")

In [ ]:
vocab = {word: i for i, (word, count) in enumerate(word_counts.items())}
index2word = {i: word for word, i in vocab.items()}

vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

In [ ]:
# Word frequencies for negative sampling
word_freq = np.array([word_counts[index2word[i]] for i in range(vocab_size)], dtype=np.float32)
word_freq = word_freq ** 0.75
word_freq = word_freq / np.sum(word_freq)  # normalize to get probabilities

# Now word_freq[i] gives the probability of sampling word i as a negative sample

In [ ]:
window_size = 8 

t = 1e-5
total_count = sum(word_counts.values())
word_prob = {word: 1 - np.sqrt(t / (count/total_count)) for word, count in word_counts.items()}

skip_gram_pairs = []

for sentence in all_sentences:
    sentence_len = len(sentence)
    for center_idx, center_word in enumerate(sentence):
        # Subsample frequent words: skip some with probability
        if np.random.rand() < word_prob[center_word]:
            continue
        
        # Determine context window boundaries
        start = max(0, center_idx - window_size)
        end = min(sentence_len, center_idx + window_size + 1)
        
        # Add (center, context) pairs
        for context_idx in range(start, end):
            if context_idx != center_idx:
                skip_gram_pairs.append((vocab[center_word], vocab[sentence[context_idx]]))

print(f"Total Skip-Gram pairs after subsampling: {len(skip_gram_pairs)}")
print(f"Example pair (indices): {skip_gram_pairs[:5]}")

In [ ]:
centers = torch.tensor([pair[0] for pair in skip_gram_pairs], dtype=torch.long)
contexts = torch.tensor([pair[1] for pair in skip_gram_pairs], dtype=torch.long)

print(centers.shape, contexts.shape)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class SGNS(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SGNS, self).__init__()
        self.embedding_dim = embedding_dim
        
        # Input embeddings (center words)
        self.in_embeddings = nn.Embedding(vocab_size, embedding_dim)
        # Output embeddings (context words)
        self.out_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Initialize embeddings
        nn.init.xavier_uniform_(self.in_embeddings.weight)
        nn.init.xavier_uniform_(self.out_embeddings.weight)
        
    def forward(self, center_words, pos_context_words, neg_context_words):
        """
        center_words: batch_size
        pos_context_words: batch_size
        neg_context_words: batch_size x K
        """
        # Get embeddings
        v_c = self.in_embeddings(center_words)              # batch x dim
        u_o = self.out_embeddings(pos_context_words)       # batch x dim
        u_k = self.out_embeddings(neg_context_words)       # batch x K x dim
        
        # Positive score
        pos_score = torch.sum(v_c * u_o, dim=1)            # batch
        pos_loss = torch.log(torch.sigmoid(pos_score) + 1e-10)  # log σ(v·u)
        
        # Negative score
        neg_score = torch.bmm(u_k.neg(), v_c.unsqueeze(2)).squeeze()  # batch x K
        neg_loss = torch.sum(torch.log(torch.sigmoid(neg_score) + 1e-10), dim=1)
        
        # Total loss
        loss = - (pos_loss + neg_loss)  # maximize likelihood
        return loss.mean()

In [ ]:
word_freq_tensor = torch.tensor(word_freq, dtype=torch.float32).to(device)  # precompute once

def get_negative_samples(batch_size, K):
    neg_samples = torch.multinomial(word_freq_tensor, batch_size*K, replacement=True)
    return neg_samples.view(batch_size, K)

In [ ]:
embedding_dim = 300
batch_size = 1024
K = 10

model = SGNS(vocab_size, embedding_dim).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.003)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(centers, contexts)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers = 4)

In [ ]:
num_epochs = 10

# Training loop
for epoch in range(num_epochs):
    total_loss = 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}")
    
    for center_batch, context_batch in pbar:
        center_batch = center_batch.to(device)
        context_batch = context_batch.to(device)
        neg_batch = get_negative_samples(len(center_batch), K).to(device)  # batch x K
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward + loss
        loss = model(center_batch, context_batch, neg_batch)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': total_loss / (pbar.n + 1)})
    
    print(f"Epoch {epoch+1} finished. Avg Loss: {total_loss/len(dataloader):.4f}")

# Save embeddings
torch.save(model.in_embeddings.weight.data.cpu(), "harry_potter_embeddings.pt")
print("Embeddings saved!")

In [ ]:
import torch
import torch.nn.functional as F

def get_nearest_words(word, model, vocab, top_k=10):
    """
    Returns top_k nearest words for a given word using trained SGNS input embeddings.
    
    Args:
        word (str): Query word.
        model (nn.Module): Trained SGNS model.
        vocab (dict): Mapping from word to index.
        top_k (int): Number of nearest neighbors to return.
    
    Returns:
        List of tuples: (word, cosine_similarity)
    """
    if word not in vocab:
        return f"'{word}' not in vocabulary"

    idx2word = {idx: w for w, idx in vocab.items()}
    model.eval()
    with torch.no_grad():
        # Use input embeddings for nearest neighbors
        word_idx = vocab[word]
        word_vec = model.in_embeddings.weight[word_idx].unsqueeze(0)  # [1, embedding_dim]
        all_embeddings = model.in_embeddings.weight  # [vocab_size, embedding_dim]

        # Cosine similarity
        cosine_sim = F.cosine_similarity(word_vec, all_embeddings)  # [vocab_size]

        # Top-k (exclude the word itself)
        top_k_idx = torch.topk(cosine_sim, top_k+1).indices.tolist()
        top_k_idx = [i for i in top_k_idx if i != word_idx][:top_k]

        neighbors = [(idx2word[i], float(cosine_sim[i])) for i in top_k_idx]

    return neighbors


def cosine_similarity_between_words(word1, word2, model, vocab):
    """
    Returns cosine similarity between two words using SGNS input embeddings.
    """
    if word1 not in vocab or word2 not in vocab:
        return f"One of the words is not in vocabulary"

    model.eval()
    with torch.no_grad():
        vec1 = model.in_embeddings.weight[vocab[word1]]
        vec2 = model.in_embeddings.weight[vocab[word2]]
        cos_sim = F.cosine_similarity(vec1.unsqueeze(0), vec2.unsqueeze(0))
    return float(cos_sim)

In [ ]:
# Top-10 neighbors of "harry"
neighbors = get_nearest_words("harry", model, vocab, top_k=10)
for w, score in neighbors:
    print(w, score)

# Cosine similarity between "harry" and "potter"
sim = cosine_similarity_between_words("harry", "cur", model, vocab)
print(f"Cosine similarity between 'harry' and 'potter': {sim:.4f}")

# CBOW

In [1]:
import requests
import os

# GitHub API URL for the folder
url = "https://api.github.com/repos/ErikaJacobs/Harry-Potter-Text-Mining/contents/Book%20Text"

# Create directory
os.makedirs("harry_potter_books", exist_ok=True)

# Get file list
response = requests.get(url)
files = response.json()

# Download only .txt files
for file in files:
    if file['name'].endswith('.txt'):
        download_url = file['download_url']
        content = requests.get(download_url).text
        
        with open(f"harry_potter_books/{file['name']}", "w", encoding="utf-8") as f:
            f.write(content)

print("All books downloaded!")

All books downloaded!


In [2]:
folder_path = "harry_potter_books"

In [3]:
file_list = sorted([f for f in os.listdir(folder_path) if f.endswith(".txt")])

In [4]:
import os
import pandas as pd

# Folder where files are downloaded

all_dfs = []

# Loop through all txt files
for file_name in file_list:
    if file_name.endswith(".txt"):
        file_path = os.path.join(folder_path, file_name)
        # Read the file assuming separator is '@' and columns are already consistent
        df = pd.read_csv(file_path, sep='@')
        all_dfs.append(df)

# Concatenate all DataFrames
df = pd.concat(all_dfs, ignore_index=True)

In [5]:
df.columns = df.columns.str.lower()

In [6]:
import re

def clean_text(text):
    text = str.lower(text)
    
    # Replace escape characters and newlines with space
    text = text.replace("\\'", "'").replace("\n", " ").replace("\r", " ")
    
    # Remove unwanted symbols (keep letters, numbers, and basic punctuation)
    text = re.sub(r"[^a-zA-Z0-9.,!?'\s]", '', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
df['text'] = df.text.apply(clean_text)

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [9]:
from nltk.tokenize import word_tokenize

sentences = df['text'].tolist()
tokenized = [word_tokenize(s.lower()) for s in sentences]

In [10]:
from collections import Counter

words = [w for sent in tokenized for w in sent]
vocab = Counter(words)

# filter rare words
min_count = 2
vocab = {w: c for w, c in vocab.items() if c >= min_count}

word2idx = {w: i for i, w in enumerate(vocab.keys())}
idx2word = {i: w for w, i in word2idx.items()}

vocab_size = len(word2idx)
print("Vocab size:", vocab_size)

Vocab size: 15924


In [11]:
window_size = 8
data = []

for sentence in tokenized:
    sentence = [w for w in sentence if w in word2idx]
    
    for i in range(window_size, len(sentence) - window_size):
        context = []
        for j in range(-window_size, window_size + 1):
            if j != 0:
                context.append(word2idx[sentence[i + j]])
        
        target = word2idx[sentence[i]]
        data.append((context, target))

In [12]:
def make_tensor(data):
    contexts = []
    targets = []
    
    for context, target in data:
        contexts.append(context)
        targets.append(target)
    
    return torch.tensor(contexts), torch.tensor(targets)

X, y = make_tensor(data)

X = X.to(device)
y = y.to(device)

In [13]:
class CBOW(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(CBOW, self).__init__()
        
        self.embeddings = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, x):
        embeds = self.embeddings(x)            # (batch, context, dim)
        mean = embeds.mean(dim=1)              # CBOW = average
        out = self.linear(mean)
        return out

In [14]:
embed_dim = 300

model = CBOW(vocab_size, embed_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
from tqdm import tqdm

epochs = 50
batch_size = 512

for epoch in range(epochs):
    total_loss = 0
    num_batches = 0

    loop = tqdm(range(0, len(X), batch_size), desc=f"Epoch {epoch+1}")

    for i in loop:
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        avg_loss = total_loss / num_batches

        # update tqdm bar
        loop.set_postfix(loss=loss.item(), avg_loss=avg_loss)

    print(f"Epoch {epoch+1} completed | Avg Loss: {avg_loss:.4f}")

Epoch 1: 100%|██████████| 2498/2498 [00:22<00:00, 112.36it/s, avg_loss=6.14, loss=5.94]


Epoch 1 completed | Avg Loss: 6.1444


Epoch 2: 100%|██████████| 2498/2498 [00:22<00:00, 111.81it/s, avg_loss=5.59, loss=5.66]


Epoch 2 completed | Avg Loss: 5.5908


Epoch 3: 100%|██████████| 2498/2498 [00:22<00:00, 109.14it/s, avg_loss=5.34, loss=5.47]


...

Epoch 30: 100%|██████████| 2498/2498 [00:23<00:00, 107.68it/s, avg_loss=3.49, loss=3.57]


Epoch 30 completed | Avg Loss: 3.4917


Epoch 31:  47%|████▋     | 1179/2498 [00:10<00:12, 107.72it/s, avg_loss=3.49, loss=3.86]IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=10000.0 (msgs/sec)
ServerApp.rate_limit_window=1.0 (secs)

Epoch 35: 100%|██████████| 2498/2498 [00:23<00:00, 107.66it/s, avg_loss=3.34, loss=3.37]


Epoch 35 completed | Avg Loss: 3.3432


Epoch 36:   8%|▊         | 209/2498 [00:01<00:21, 107.07it/s, avg_loss=3.51, loss=3.54]

In [ ]:
embeddings = model.embeddings.weight.data

def get_vector(word):
    return embeddings[word2idx[word]].cpu().numpy()

In [ ]:
import torch.nn.functional as F

def most_similar(word, top_k=5):
    vec = embeddings[word2idx[word]]
    
    sims = F.cosine_similarity(vec.unsqueeze(0), embeddings)
    top = torch.topk(sims, top_k + 1).indices.tolist()
    
    return [idx2word[i] for i in top if idx2word[i] != word][:top_k]

print(most_similar("harry"))